# Semiconductor Equipment Reliability and Anomaly Detection: Data Ingestion and Cleaning

For this project, I am using **SECOM Manufacturing Dataset (UCI)**. This dataset consists of 590 sensors data for 1567 wafer runs. Since a complex modern semi-conoductor manufacturing process is normally under consistent surveillance through the monitoring of signal/variables collectes from sensors and process measurement points. 

Some of these signals are trivial. It could consist of irrelevant information or noise. Usually, the useful information is hidden in the irrevelant information or noise. We can apply feature selection to determine most relevant signals. These signals could then be used by Process Engineers to determine key factors contributing to yield excursions downstream in the process. 

This process decreases time to learning and reduce per unit production costs.

My goal for this step is to:
- load raw sensor data
- understand its structure
- Split train/test sets
- handle midding values
- Save cleaned data 

### Loading the Raw Sensor Data + Understanding Structure

In [1]:
import pandas as pd
import numpy as np

# Loading the dataset
df = pd.read_csv('../data/uci-secom.csv')

print(f"Shape: {df.shape}")
df.head()

Shape: (1567, 592)


,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


In [2]:
# Checking the Pass/Fail values
print(df['Pass/Fail'].value_counts())

Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64


With the result above, we see that the Pass values are over ten times the Fail values. This is called **class imbalance**. 

This means the the model could predict 'pass' everytime and be over 90% accurate while missing every fail result. This is a problem.

In [3]:
# Checking missing values
print(df.isnull().sum())
print(df.isnull().mean()*100)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')
print(f'Missing values percentage: {df.isnull().mean().mean()*100}')

Time          0
0             6
1             7
2            14
3            14
             ..
586           1
587           1
588           1
589           1
Pass/Fail     0
Length: 592, dtype: int64
Time         0.000000
0            0.382897
1            0.446713
2            0.893427
3            0.893427
               ...   
586          0.063816
587          0.063816
588          0.063816
589          0.063816
Pass/Fail    0.000000
Length: 592, dtype: float64

Total missing values: 41951
Missing values percentage: 4.522219251798065


Although, there are a lot of missing values, the overall percentage of missing values is just 4.5%. 

This means the sensor in the fab could mean the sensor was offline, the reading was out of range, or maybe tool malfunctioned. 

### Seperating Features and Labels + Train/Test Split

Here, we will split first and then handle missing values. 

If I handle the missing values on 'all' of the dataset, then it creates **data leakage**. That creates bias when testing the data. 

In [4]:
from sklearn.model_selection import train_test_split

# Seperating features (sensors) and label
X = df.drop(columns = ['Pass/Fail', 'Time'])
y = df['Pass/Fail']

# Checking shapes
print(f"X Shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Values in y: {y.unique()}")

# Splitting training/testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 29
)

print(f"\nTraining set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

X Shape: (1567, 590)
y shape: (1567,)
Values in y: [-1  1]

Training set: (1253, 590)
Testing set: (314, 590)


We have confirmed that the features (X) only consists of the sensors data. We also verified that the label (y) consists of just Pass and Fail values.

### Handling missing values

We have four choices to handle the missing values:
1. Mean: Replacing the missing values with the avergae values. However, the mean gets pulled by extreme values which are common in sensor data.
2. Median: Replacing the missing values with the middle values (median). Median makes sure it does not get pulled by the extreme values. We do lose some precision.
3. Mode: Replacing the missing values with the most common values. It works better for categorical data. Since sensor readings are continuous, this does not apply here.
4. Drop missing values: This is dangerous because we already have class imbalance. We might accidentally drop failure cases. With small number of failure cases, each row is important.

Final decision: Median

In [5]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy = 'median')

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Confirm if any missing values remain
print(pd.isnull(X_train_imputed).sum().sum())
print(pd.isnull(X_test_imputed).sum().sum())

0
0


We have verified successfully that no missing values remain.

Notice that I imputed the training set and testing set seperately. Moreover, I used `fit_transform` for the training set and `transform` for the testing set. The difference between the two is that the former calculates and remembers the median, while the latter applies the already-remembered median directly without calculating it seperately. This ensures no **data leakage**.

### Removing Zero-Variance Features

In this section, we will remove the sensors that have zero variance. This is because they output the same value for every single wafer run. This tells us nothing about why a wafer failed. 

In [6]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

X_train_selected = selector.fit_transform(X_train_imputed)
X_test_selected = selector.transform(X_test_imputed)

# Checking the number of sensors removed
original = X_train_imputed.shape[1]
remaining = X_train_selected.shape[1]

print(f"Number of sensors removed: {original - remaining}")

Number of sensors removed: 116


This result shows that 116 sensors, which is a big chunk, had a zero-variance and did not give us any information about why a wafer failed.

Important note: removing these sensors does NOT mean that the sensors are trivial. For some cases, the sensors are EXPECTED to have a zero-variance. However, for our case, we need the a variance > 0 to help identify the causes for a wafer failure. A zero-variance cannot help us mathematically identify the difference between passing or failing for the wafer. 

### Saving Cleaned Data

In [7]:
X_train_clean = pd.DataFrame(X_train_selected)
X_test_clean = pd.DataFrame(X_test_selected)

X_train_clean.to_csv('../data/X_train_clean.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
X_test_clean.to_csv('../data/X_test_clean.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

print(f"X_train: {X_train_clean.shape}")
print(f"X_test: {X_test_clean.shape}")

X_train: (1253, 474)
X_test: (314, 474)


This is the final step of Data Ingestion and cleaning. We have verified that the number of rows stay consistent for training and testing datasets. The zero-variance sensors are now removed. We can move to the next step - **Exploratory Data Analysis**.